# Proyecto de PLN: Normalización, Vectorización y Semántica Distribucional
## Caso de Estudio: Don Quijote de la Mancha (Miguel de Cervantes)

Pipeline completo de Procesamiento de Lenguaje Natural:
1. **Tokenización** — descomposición en tokens con spaCy.
2. **Filtrado de Ruido** — eliminación de stop words y puntuación.
3. **Lematización y Normalización** — lemas canónicos en minúsculas.
4. **Stemming vs Lematización** — comparativa NLTK / spaCy.
5. **Reducción de Dimensionalidad** — contracción del vocabulario.
6. **Vectorización BoW y TF-IDF** — representaciones numéricas dispersas.
7. **Espacio Vectorial 3D** — PCA sobre matrices BoW / TF-IDF.
8. **Semántica Distribucional** — GloVe pre-entrenado y Word2Vec propio.


In [ ]:
import os
import re
import pandas as pd
import spacy
from nltk.stem import SnowballStemmer

print("Librerías importadas correctamente.")


In [ ]:
# Cargar modelo en español de spaCy
try:
    nlp = spacy.load("es_core_news_sm")
    print("Modelo es_core_news_sm cargado exitosamente.")
except OSError:
    print("Descargando modelo...")
    from spacy.cli import download
    download("es_core_news_sm")
    nlp = spacy.load("es_core_news_sm")


In [ ]:
# Carga del libro y extracción del Capítulo Primero
with open("don_quijote.txt", "r", encoding="utf-8") as f:
    texto_completo = f.read()

patron = r"Capítulo primero\..*?(?=Capítulo II\.|\Z)"
match = re.search(patron, texto_completo, re.DOTALL | re.IGNORECASE)
texto_cap1 = match.group(0).strip() if match else texto_completo[:10000]

print(f"Texto cargado. Longitud: {len(texto_cap1)} caracteres.")
print(f"Fragmento inicial:\n{texto_cap1[:250]}...")


In [ ]:
# 1. TOKENIZACIÓN
doc = nlp(texto_cap1)

print("--- 1. Tokenización ---")
print(f"Total de tokens generados: {len(doc)}")
primeros_tokens = [token.text for token in doc if not token.is_space][:20]
print(f"Muestra de primeros 20 tokens:\n{primeros_tokens}")


In [ ]:
# 2. FILTRADO DE STOP WORDS Y RUIDO
tokens_relevantes = []
tokens_ruido = []

for token in doc:
    if not token.is_stop and not token.is_punct and not token.is_space and token.text.strip():
        tokens_relevantes.append(token.text)
    elif token.is_stop or token.is_punct:
        tokens_ruido.append(token.text)

print("--- 2. Filtrado de Stop Words y Puntuación ---")
print(f"Tokens eliminados (Ruido): {len(tokens_ruido)}")
print(f"Tokens conservados (Contenido útil): {len(tokens_relevantes)}")
print(f"Muestra eliminadas: {tokens_ruido[:10]}")
print(f"Muestra conservadas: {tokens_relevantes[:10]}")


In [ ]:
# 3. LEMATIZACIÓN Y NORMALIZACIÓN
tokens_normalizados = []
cambios_interesantes = []

for token in doc:
    if not token.is_stop and not token.is_punct and not token.is_space and token.text.strip():
        lema = token.lemma_.lower()
        tokens_normalizados.append(lema)
        if token.text.lower() != lema and len(cambios_interesantes) < 10:
            cambios_interesantes.append(f"{token.text} -> {lema}")

print("--- 3. Lematización y Normalización ---")
print(f"Total de tokens normalizados: {len(tokens_normalizados)}")
print("Transformaciones morfológicas (Original -> Lema):")
for c in cambios_interesantes:
    print(f"  * {c}")
print(f"\nMuestra de tokens finales: {tokens_normalizados[:10]}")


In [ ]:
# 4. COMPARATIVA: STEMMING VS LEMATIZACIÓN
stemmer = SnowballStemmer("spanish")
data_comparativa = []

for token in doc:
    if not token.is_punct and not token.is_space and not token.is_stop and token.text.strip():
        raiz_stem = stemmer.stem(token.text)
        lema = token.lemma_.lower()
        data_comparativa.append({
            "Original": token.text,
            "Stemming (NLTK)": raiz_stem,
            "Lematización (spaCy)": lema,
            "Coinciden": raiz_stem == lema
        })

df_comparativa = pd.DataFrame(data_comparativa)
palabras_clave = ["acordarme", "vivía", "antigua", "corredor", "leyendo", "imaginación", "deseaba", "caballeros", "hicieron", "podía"]
filtro = df_comparativa[df_comparativa["Original"].str.lower().isin(palabras_clave)].drop_duplicates(subset=["Original"])

print("--- Comparativa en Palabras Representativas ---")
print(filtro.to_string(index=False))
print("\n--- Primeros 10 Tokens ---")
print(df_comparativa.head(10).to_string(index=False))


In [ ]:
# 5. REDUCCIÓN DE DIMENSIONALIDAD DEL VOCABULARIO
vocabulario_original = len(set([t.text.lower() for t in doc if not t.is_punct and not t.is_space]))
vocabulario_lemas = len(set(tokens_normalizados))
reduccion = ((vocabulario_original - vocabulario_lemas) / vocabulario_original) * 100

resumen = pd.DataFrame({
    "Métrica": ["Tokens Totales", "Tokens Útiles (Sin Ruido)", "Vocabulario Único Original", "Vocabulario Único Lematizado", "Reducción de Dimensionalidad"],
    "Valor": [f"{len(doc):,}", f"{len(tokens_normalizados):,}", f"{vocabulario_original:,}", f"{vocabulario_lemas:,}", f"{reduccion:.2f}%"]
})
print("--- Resumen de Reducción de Dimensionalidad ---")
print(resumen.to_string(index=False))


---
## Checkpoint 4: Representación Vectorial y Semántica

| Modelo | Idea clave | Ventaja | Desventaja |
|---|---|---|---|
| **Bag-of-Words** | Conteos de términos | Simple, efectivo | Ignora orden; alta sparsity |
| **TF-IDF** | Premia términos distintivos del documento | Captura relevancia relativa | Igual pierde el orden |


In [ ]:
# 6. CORPUS LEMATIZADO POR ORACIONES
corpus_lematizado = []

for oracion in doc.sents:
    lemas_oracion = [
        token.lemma_.lower()
        for token in oracion
        if not token.is_punct and not token.is_space and not token.is_stop
    ]
    if lemas_oracion:
        corpus_lematizado.append(" ".join(lemas_oracion))

print(f"Total de oraciones procesadas: {len(corpus_lematizado)}")
print(f"\nPrimera oración del corpus:")
print(f"  '{corpus_lematizado[0]}'")


In [ ]:
# 7. BAG-OF-WORDS Y TF-IDF
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

bow_vectorizer = CountVectorizer()
X_bow = bow_vectorizer.fit_transform(corpus_lematizado)

tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(corpus_lematizado)

print(f"Forma de la matriz BoW:    {X_bow.shape}  (oraciones x términos)")
print(f"Forma de la matriz TF-IDF: {X_tfidf.shape}  (oraciones x términos)")
print(f"Vocabulario: {len(bow_vectorizer.vocabulary_)} términos únicos")
print(f"Densidad BoW (no-ceros/total): {X_bow.nnz / (X_bow.shape[0] * X_bow.shape[1]):.4f}")

vocab_tfidf = tfidf_vectorizer.get_feature_names_out()
primera = X_tfidf[0].toarray()[0]
top_indices = np.argsort(primera)[::-1][:10]
print(f"\nTop 10 términos TF-IDF (primera oración):")
for i in top_indices:
    if primera[i] > 0:
        print(f"  {vocab_tfidf[i]:<20} peso: {primera[i]:.4f}")


In [ ]:
# 8. VISUALIZACIÓN 3D: BoW vs TF-IDF con PCA
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA

def graficar_palabras_3d(ax, matriz, vocabulario, titulo, color_puntos):
    matriz_palabras = matriz.T
    pca = PCA(n_components=3)
    coords = pca.fit_transform(matriz_palabras.toarray())
    x, y, z = coords[:, 0], coords[:, 1], coords[:, 2]
    ax.scatter(x, y, z, c=color_puntos, s=80, edgecolors='k', alpha=0.8, depthshade=True)
    for i, palabra in enumerate(vocabulario[:30]):
        ax.text(x[i], y[i], z[i] + 0.01, palabra, fontsize=7)
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_xlabel('Comp. Principal 1')
    ax.set_ylabel('Comp. Principal 2')
    ax.set_zlabel('Comp. Principal 3')
    ax.plot([0,0],[0,0],[z.min(),z.max()], c='grey', ls='--', lw=0.5, alpha=0.3)
    ax.plot([x.min(),x.max()],[0,0],[0,0], c='grey', ls='--', lw=0.5, alpha=0.3)
    ax.plot([0,0],[y.min(),y.max()],[0,0], c='grey', ls='--', lw=0.5, alpha=0.3)

fig = plt.figure(figsize=(18, 8))
fig.suptitle('Representación Vectorial del Quijote — Cap. 1', fontsize=14)
ax1 = fig.add_subplot(121, projection='3d')
vocab_bow = bow_vectorizer.get_feature_names_out()
graficar_palabras_3d(ax1, X_bow, vocab_bow, 'Espacio BoW 3D (Conteos)', 'orange')
ax2 = fig.add_subplot(122, projection='3d')
graficar_palabras_3d(ax2, X_tfidf, vocab_tfidf, 'Espacio TF-IDF 3D (Importancia)', 'teal')
plt.tight_layout()
plt.savefig('espacio_vectorial_3d.png', dpi=150, bbox_inches='tight')
plt.show()
print("Guardado: espacio_vectorial_3d.png")


---
## Checkpoint 5: Semántica Distribucional

Los vectores anteriores (BoW, TF-IDF) eran **dispersos** — miles de dimensiones, casi todo cero.
La semántica distribucional produce vectores **densos** (50-300 dimensiones, todo números reales) que capturan significado.

**Hipótesis distribucional**: *«Una palabra es conocida por la compañía que frecuenta»*.
Si «caballero» e «hidalgo» aparecen en contextos similares, sus vectores estarán cerca.

| Enfoque | Algoritmo | Idea |
|---|---|---|
| Conteo + factorización | **GloVe** | Matriz global de co-ocurrencias → compresión matemática |
| Predicción (red neuronal) | **Word2Vec** | CBOW (adivina la palabra) / Skip-gram (adivina el contexto) |


In [ ]:
# 9. EMBEDDINGS PRE-ENTRENADOS: GLOVE 50d
# GloVe entrenado sobre Wikipedia + Gigaword (6B tokens, 50 dimensiones)
import gensim.downloader as api

print("Cargando modelo GloVe 50d (primera vez descarga ~66 MB)...")
glove = api.load("glove-wiki-gigaword-50")
print(f"Modelo cargado. Vocabulario: {len(glove)} palabras.")

# Visualizar embeddings de palabras semánticamente relacionadas
words = ["man", "woman", "boy", "girl", "king", "queen"]
vecs = np.array([glove[w] for w in words])

plt.figure(figsize=(14, 4))
plt.imshow(vecs, cmap="RdBu_r", aspect="auto")
plt.yticks(range(len(words)), words, fontsize=13)
plt.xlabel("Dimensión (1-50)")
plt.title("Embeddings GloVe — vectores densos de 50 dimensiones")
plt.colorbar(label="Valor")
plt.tight_layout()
plt.savefig("glove_heatmap.png", dpi=150)
plt.show()
print("Guardado: glove_heatmap.png")


In [ ]:
# 10. ARITMÉTICA SEMÁNTICA: king - man + woman ≈ queen
from sklearn.metrics.pairwise import cosine_similarity

king = glove["king"]
man = glove["man"]
woman = glove["woman"]
queen = glove["queen"]
resultado = king - man + woman

labels = ["king", "- man", "+ woman", "= resultado", "queen (real)"]
vecs_analogia = np.array([king, -man, woman, resultado, queen])

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(vecs_analogia, cmap="RdBu_r", aspect="auto")
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=13, fontweight="bold")
ax.set_xlabel("Dimensión")
ax.set_title("Analogía: king - man + woman ≈ queen", fontsize=14, fontweight="bold")
plt.colorbar(im, ax=ax, label="Valor")

sim = cosine_similarity([resultado], [queen])[0][0]
ax.annotate(
    f"similitud coseno(resultado, queen) = {sim:.4f}",
    xy=(0.5, -0.22), xycoords="axes fraction",
    ha="center", fontsize=12, fontstyle="italic"
)
plt.tight_layout()
plt.savefig("glove_analogy.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Similitud coseno resultado vs queen: {sim:.4f}")
print("Guardado: glove_analogy.png")


In [ ]:
# 11. WORD2VEC DESDE CERO — Don Quijote de la Mancha
# Usamos el libro completo para tener suficientes datos de entrenamiento
import multiprocessing
from gensim.models import Word2Vec

# Procesar los primeros 500K caracteres del Quijote completo
doc_completo = nlp(texto_completo[:500000])

sentences_w2v = []
for sent in doc_completo.sents:
    tokens = [
        token.lemma_.lower()
        for token in sent
        if not token.is_stop and not token.is_punct and token.text.strip()
    ]
    if len(tokens) > 1:
        sentences_w2v.append(tokens)

print(f"Oraciones para entrenar: {len(sentences_w2v)}")
print(f"Ejemplo (primeros tokens): {sentences_w2v[0][:8]}")

# Skip-gram (sg=1): predice el contexto dado la palabra central
print("\nEntrenando Word2Vec Skip-gram...")
w2v_model = Word2Vec(
    sentences_w2v,
    vector_size=50,
    window=5,
    min_count=3,
    workers=multiprocessing.cpu_count(),
    sg=1,
    seed=42
)
print(f"Vocabulario aprendido: {len(w2v_model.wv)} palabras")
print("\nVector de 50 dimensiones de 'caballero':")
try:
    print(w2v_model.wv["caballero"])
except KeyError:
    print("'caballero' no en vocabulario")


In [ ]:
# 12. EXPLORACIÓN SEMÁNTICA — palabras más cercanas

def mostrar_similares(palabra, modelo, topn=5):
    try:
        similares = modelo.wv.most_similar(palabra, topn=topn)
        print(f"\nPalabras más cercanas a '{palabra}':")
        for w, score in similares:
            print(f"  {w:<20} similitud: {score:.4f}")
    except KeyError:
        print(f"'{palabra}' no está en el vocabulario.")


# Explorar personajes y conceptos del Quijote
for palabra in ["caballero", "hidalgo", "espada", "batalla", "amor"]:
    mostrar_similares(palabra, w2v_model)


In [ ]:
# 13. VISUALIZACIÓN 3D DEL ESPACIO SEMÁNTICO (Word2Vec)

# Top 80 palabras más frecuentes del vocabulario
vocabulario_w2v = list(w2v_model.wv.index_to_key)[:80]
vectores_w2v = w2v_model.wv[vocabulario_w2v]

# Reducir de 50 a 3 dimensiones
pca_w2v = PCA(n_components=3)
coords_3d = pca_w2v.fit_transform(vectores_w2v)

df_w2v = pd.DataFrame(coords_3d, columns=["x", "y", "z"])
df_w2v["palabra"] = vocabulario_w2v

fig = plt.figure(figsize=(14, 9))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(df_w2v["x"], df_w2v["y"], df_w2v["z"],
           c="crimson", s=80, edgecolors="white", alpha=0.8)

for _, row in df_w2v.head(30).iterrows():
    ax.text(row["x"], row["y"], row["z"], f" {row['palabra']}", size=9)

ax.set_title("Espacio Semántico Word2Vec — Don Quijote de la Mancha",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Dimensión Latente 1")
ax.set_ylabel("Dimensión Latente 2")
ax.set_zlabel("Dimensión Latente 3")
plt.tight_layout()
plt.savefig("word2vec_3d.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: word2vec_3d.png")


## Conclusiones Técnicas

**Checkpoint 2 — Normalización:**
1. El filtrado de stop words redujo el ruido preservando el contenido semántico.
2. La lematización es superior al stemming: produce formas válidas del diccionario.
3. La normalización redujo el vocabulario único en más del 30%.

**Checkpoint 4 — Vectorización:**
4. BoW y TF-IDF transforman texto en vectores numéricos dispersos operables con álgebra lineal.
5. TF-IDF premia los términos distintivos de cada documento y penaliza los omnipresentes.
6. La sparsity de ambas matrices justifica el uso de `scipy.sparse`.

**Checkpoint 5 — Semántica Distribucional:**
7. Word2Vec produce vectores **densos** de 50 dimensiones que capturan relaciones semánticas.
8. La hipótesis distribucional funciona: «caballero» e «hidalgo» están cerca en el espacio vectorial.
9. GloVe demuestra aritmética semántica: `king - man + woman ≈ queen` (similitud coseno > 0.85).
10. La calidad de los embeddings escala con el volumen de datos — el libro completo mejora las asociaciones.
